# Frozen MPNet: window coverage follow-up

Experiment log. One inference correction: overlapping 382-token windows, stride 286, maximum classifier score. Encoder and classifier training inputs stay fixed. Classifier seeds 17, 42, 73; validation-only thresholds.

The companion `window_followup_inputs.tar.gz` contains the frozen protocol, original train/validation rows, screened unused holdout, and exact scripts. Upload it to Colab session storage. The holdout is not read for fitting or threshold selection.

HTML attribute/comment loss, source imbalance and synthetic template defects remain limitations. This is research evidence, not a standalone security gate.

In [ ]:
%pip uninstall -y torchvision torchaudio librosa
%pip install -q torch==2.9.1 sentence-transformers==5.1.2 transformers==4.57.3 datasets==4.4.1 xgboost==3.1.2 scikit-learn==1.7.2 numpy==2.3.5 scipy==1.16.3 beautifulsoup4==4.14.2 accelerate==1.11.0

In [ ]:
import os, subprocess, importlib.metadata
expected = {'torch':'2.9.1','sentence-transformers':'5.1.2','transformers':'4.57.3','xgboost':'3.1.2','numpy':'2.3.5','scikit-learn':'1.7.2','datasets':'4.4.1'}
for name, version in expected.items():
    assert importlib.metadata.version(name).split('+')[0] == version, (name, importlib.metadata.version(name))
os.environ.update(USE_TF='0', USE_FLAX='0', WANDB_MODE='disabled')
print('Pinned experiment environment verified.')


In [ ]:
from google.colab import files
files.upload()

In [ ]:
from pathlib import Path
import hashlib, tarfile, json
archive = Path('/content/window_followup_inputs.tar.gz')
assert hashlib.sha256(archive.read_bytes()).hexdigest() == '82a34ca52b073d54b7a62d40aed34adef40ea282c00b14b17983d2981f4d9d24'
output = Path('/content/window_followup')
if not output.exists():
    with tarfile.open(archive) as bundle:
        bundle.extractall('/content', filter='data')
print('Frozen protocol:', (output / 'protocol.json').read_text())


In [ ]:
import os, subprocess
for stage in ('validate', 'evaluate'):
    command = ['python','-u','/content/window_followup/windowed_followup.py',stage,'--output','/content/window_followup']
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=dict(os.environ,USE_TF='0',USE_FLAX='0',WANDB_MODE='disabled')) as process:
        for line in process.stdout:
            print(line,end='',flush=True)
        if process.wait():
            raise RuntimeError(stage + ' failed; final evaluation must not proceed on failure')


In [ ]:
import json
from pathlib import Path
print(Path('/content/window_followup/holdout_results.json').read_text())

In [ ]:
import tarfile
from google.colab import files
with tarfile.open('/content/window_followup_results.tar.gz','w:gz') as archive:
    archive.add('/content/window_followup',arcname='window_followup')
files.download('/content/window_followup_results.tar.gz')